# SEQUENTIAL GENERATION OF IMAGES FROM TEXT


# IMPORT LIBRARIES AND INSERT STABILITY API KEY


In [ ]:
from io import BytesIO
import IPython
import json
import os
from PIL import Image
import requests
import time
import getpass
from google import genai #LLM
from google.genai.errors import APIError 


try:
    from google.colab import output
except Exception:
    class _ColabOutputFallback:
        def no_vertical_scroll(self): 
            pass
    output = _ColabOutputFallback()

STABILITY_KEY = getpass.getpass('Enter your API Key') 


# INSERT GEMINI API KEY

In [2]:
GEMINI_KEY = getpass.getpass('Enter your GEMINI API Key')
client = genai.Client(api_key=GEMINI_KEY)
LLM_MODEL = 'gemini-2.5-flash'

# STABILITY GENERATION REQUIRED FUNCTION

In [ ]:

def send_generation_request(
    host,
    params,
):
    headers = {
        "Accept": "image/*",
        "Authorization": f"Bearer {STABILITY_KEY}"
    }

    files = {}
    image = params.pop("image", None)
    mask = params.pop("mask", None)
    if image is not None and image != '':
        files["image"] = open(image, 'rb')
    if mask is not None and mask != '':
        files["mask"] = open(mask, 'rb')
    if len(files)==0:
        files["none"] = ''

    print(f"Sending REST request to {host}...")
    response = requests.post(
        host,
        headers=headers,
        files=files,
        data=params
    )
    if not response.ok:
        raise Exception(f"HTTP {response.status_code}: {response.text}")

    return response


# FUNZIONE GENERAZIONE PROMPT VISIVO ZERO SHOTS

system_prompt is the only one that changes between the three versions of exercise A, here is the experiment list 1A, 2A and 3A


#USED FOR 3C:

system_prompt = (
        "You are a **Creative Director** specialized in generating visual prompts for Young Adult Graphic Novels, Animated Series, and Adventure Video Games (target age: 10-15 years old). "
        "Your primary goal is to translate narrative sentences into **shigh-energy, cinematic, and deeply atmospheric visual prompts that capture dynamic action, emotional intensity, and complex world-building** suitable for a teenage audience. \n\n"
        
        "**Core Mandate: Continuity and Character Consistency:** "
        "You must guarantee **perfect continuity** of characters, creatures, and the environment's style across the entire story sequence. "
        "Use the `Context of previous scenes` to ensure the character's appearance (face, clothes, size) and the environment remain **perfectly identical** from scene to scene. \n\n"
        
        "**Mandatory Style Requirements:** Always include these elements: "
        "**Dynamic Concept Art Style,** **Atmospheric & Dramatic Lighting** (e.g., strong rim lighting, volumetric rays, cinematic contrast), "
        "**Consistent Comicbook Design** (highly detailed, expressive characters), Complex Composition(dynamic angles, sense of motion). "
        
        "**Content Rules:** Focus on describing clear **actions, complex color palettes and sophisticated emotions**(e.g., stoicism, gritty determination, quiet focus ). "
        "Describe dynamic perspective and camera work (e.g., 'Medium shot, low angle to emphasize power). "
        "Do not use complex cinematic terms, abstract language, or sharp contrast. The mood must always be **positive and inviting** unless the narrative explicitly requires a momentary surprise or gentle suspense. \n\n"
        
        "**The final output must be ONLY the visual prompt text, with no extra commentary.**"
    
        )

#USED FOR 2C:

system_prompt = (
        "You are a **Creative Director** specialized in generating visual prompts for children's storybooks and animated movies (target age: 5-10 years old). "
        "Your primary goal is to translate narrative sentences into **simple, colorful, and highly descriptive visual prompts** that are immediately engaging for kids. \n\n"
        
        "**Core Mandate: Continuity and Character Consistency:** "
        "You must guarantee **perfect continuity** of characters, creatures, and the environment's style across the entire story sequence. "
        "Use the `Context of previous scenes` to ensure the character's appearance (face, clothes, size) and the environment remain **perfectly identical** from scene to scene. \n\n"
        
        "**Mandatory Style Requirements:** Always include these elements: "
        "**Vibrant Digital Illustration,** **Playful & Bright Lighting** (e.g., warm, sunny, magical glow), "
        "**Consistent Cartoon/Storybook Design** (charming, friendly look), simple clear composition, ultra-detailed textures. "
        
        "**Content Rules:** Focus on describing clear **actions, primary emotions** (happy, curious, surprised), and **vivid colors**. "
        "Describe changes in simple perspective (close-up, wide shot) and light source (e.g., 'a sparkle of magic'). "
        "Do not use complex cinematic terms, abstract language, or sharp contrast. The mood must always be **positive and inviting** unless the narrative explicitly requires a momentary surprise or gentle suspense. \n\n"
        
        "**The final output must be ONLY the visual prompt text, with no extra commentary.**"
        )

#USED FOR 1C:

system_prompt = ()

In [ ]:

def generate_visual_prompt(narrative_sentence: str, previous_scenes_context: str) -> str:
    if 'client' not in globals() or client is None:
        print("Client LLM non inizializzato.")
        return narrative_sentence

    # system prompt based on your specifications to ensure coherence and detail

    system_prompt = (
    "You are a **Creative Director** specialized in generating visual prompts for Young Adult Graphic Novels, Animated Series, and Adventure Video Games (target age: 10-15 years old). "
    "Your primary goal is to translate narrative sentences into **shigh-energy, cinematic, and deeply atmospheric visual prompts that capture dynamic action, emotional intensity, and complex world-building** suitable for a teenage audience. \n\n"
    
    "**Core Mandate: Continuity and Character Consistency:** "
    "You must guarantee **perfect continuity** of characters, creatures, and the environment's style across the entire story sequence. "
    "Use the `Context of previous scenes` to ensure the character's appearance (face, clothes, size) and the environment remain **perfectly identical** from scene to scene. \n\n"
    
    "**Mandatory Style Requirements:** Always include these elements: "
    "**Dynamic Concept Art Style,** **Atmospheric & Dramatic Lighting** (e.g., strong rim lighting, volumetric rays, cinematic contrast), "
    "**Consistent Comicbook Design** (highly detailed, expressive characters), Complex Composition(dynamic angles, sense of motion). "
    
    "**Content Rules:** Focus on describing clear **actions, complex color palettes and sophisticated emotions**(e.g., stoicism, gritty determination, quiet focus ). "
    "Describe dynamic perspective and camera work (e.g., 'Medium shot, low angle to emphasize power). "
    "Do not use complex cinematic terms, abstract language, or sharp contrast. The mood must always be **positive and inviting** unless the narrative explicitly requires a momentary surprise or gentle suspense. \n\n"
    
    "**The final output must be ONLY the visual prompt text, with no extra commentary.**"
        
    )

    if previous_scenes_context:
        context_instruction = f"Maintain visual and contextual coherence with previous scenes. Context of previous scenes: {previous_scenes_context}"
    else:
        context_instruction = "This is the first scene. Ensure the style is established clearly."
        
    user_prompt_content = f"{context_instruction}\n\nSentence to represent: \"{narrative_sentence}\""

    full_prompt = f"{system_prompt}\n\n--- TASK ---\n\n{user_prompt_content}"


    # This function includes retry logic for handling temporary API errors
    MAX_RETRIES = 5  
    BASE_DELAY = 5

    for attempt in range(MAX_RETRIES):
            try:
                
                response = client.models.generate_content(
                    model=LLM_MODEL,
                    contents=[
                        {"role": "user", "parts": [{"text": full_prompt}]}
                    ]
                )
                
                return response.text.strip()
                
            except APIError as e:
                
                if '503 UNAVAILABLE' not in str(e) and '429' not in str(e):
                    print(f"IRRECOVERABLE LLM API Error on attempt {attempt + 1}/{MAX_RETRIES}: {e}. Interruption.")
                    return narrative_sentence 
                    
                print(f"TEMPORARY LLM API Error (503/429) on attempt {attempt + 1}/{MAX_RETRIES}. Error: {e}")
                
                if attempt < MAX_RETRIES - 1:
                    delay = BASE_DELAY * (2 ** attempt)
                    print(f"Waiting for {delay} seconds before the next attempt...")
                    time.sleep(delay)
                
    print(f"CRITICAL ERROR: Prompt generation failed after {MAX_RETRIES} attempts.")
    return narrative_sentence 



# TEXT REWRITE FUNCTION



Function used only in the 1C,2C,3C experiments, where the LLM rewrite the story to use it as a new baseline for the prompts.

In [ ]:
REWRITE_MODE = True 
REWRITTEN_PROMPTS_PATH = # Directory path where rewritten prompts will be saved example: "directory/to/save/prompts"
os.makedirs(REWRITTEN_PROMPTS_PATH, exist_ok=True) 
 
MAX_RETRIES = 5  
BASE_DELAY = 5   

def rewrite_story_for_image_creation(story):
    
    
    prompt = f"""
    Rewrite the following story into a concise and descriptive text that can be easily used to generate a sequence of immage with coherence and high precision.
    
    CRUCIAL INSTRUCTION: The rewritten story MUST be divided into EXACTLY 5 distinct scenes with maximum of 75 English words per scene. 
    Each scene must be visually descriptive and separated from the next scene by a double newline:
    
    Focus on high coherence, visual elements, characters, settings, and actions. Avoid abstract concepts and keep sentences direct. Focus on making each scene the more accurate

    Story: {story}

    Rewritten for image generation (5 Scenes, separated by )
    """
    response = client.models.generate_content(model=LLM_MODEL,
        contents=[{"role": "user", "parts": [{"text": prompt}]}])
    

    for attempt in range(MAX_RETRIES):
            try:
                response = client.models.generate_content(
                    model=LLM_MODEL,
                    contents=[{"role": "user", "parts": [{"text": prompt}]}]
                )
                return response.text.strip()
                
            except APIError as e:
                error_message = str(e)
                if '503' not in error_message and '429' not in error_message:
                    print(f"LLM API IRRECOVERABLE Error on attempt {attempt + 1}: {e}. Break.")
                    break 
                    
                print(f"LLM API TEMPORARY Error (503/429) on attempt {attempt + 1}. Error: {e}")
                
                if attempt < MAX_RETRIES - 1:
                    delay = BASE_DELAY * (2 ** attempt)
                    print(f"Waiting for {delay} seconds before the next attempt...")
                    time.sleep(delay)
            
    print(f"CRITICAL ERROR: History rewrite failed after {MAX_RETRIES} attempts. Returning the original story as a fallback.")
    return f"Rewriting failed: {story}"


# TEXT TO IMAGE AND IMAGE TO IMAGE

You need to specify the directory for the "STORY_BASE_PATH" where the stories are located.
You need to specify the directory for the "REWRITE_STORY_PATH" where the stories rewrited are gonna be located.

WARNING: in the line for story_number in range(3, 9), you need to specify what story do you want to generate among your present stories. For example if you want to generate stories from 3 to 8 (i need to specify the last one as i+1).

In [ ]:
ORIGINAL_STORIES_PATH = # directory path where original stories are stored example: "directory/of/original/stories"
REWRITE_STORY_PATH = # directory path where rewritten stories will be saved example: "directory/to/save/rewritten/stories"

if REWRITE_MODE:
    STORY_BASE_PATH = REWRITE_STORY_PATH
else:
    STORY_BASE_PATH = ORIGINAL_STORIES_PATH
    
VLM_HOST = "https://api.stability.ai/v2beta/stable-image/generate/sd3"
VLM_MODEL = "sd3.5-medium"
I2I_STRENGTH = 0.75
ASPECT_RATIO = "1:1" 
NEGATIVE_PROMPT_GLOBAL = "text, writing, words, signature, watermark, logo, cropped, blurry, distorted, low quality, worst quality, low resolution, noise, jpeg artifacts, grain, ugly, oversaturated, disfigured, deformed, mutation, extra limbs, missing limbs, weird eyes, multiple wizards, duplicate person, inconsistent character, changing body position, changing face, poorly drawn face, bad anatomy, malformed, out of frame, extra fingers, too many hands, tiling, cartoon, sketch, painting, illustration, drawing, plastic, toy, figurine, sculpture, fake, artificial, dry skin, dull water, flat light, monochrome, bad blending, artifacts, visual noise, distorted cat"

all_stories_data = [] # To store data for all stories for metrics calculation


# WARNING: Specifiy the range of stories to process example from 1 to 31 for stories 1.txt to 30.txt
for story_number in range(1, 31): 
    STORY_FILE_NAME = f"{story_number:02d}.txt" 
    original_story_path = os.path.join(ORIGINAL_STORIES_PATH, STORY_FILE_NAME)
    rewritten_story_path = os.path.join(REWRITE_STORY_PATH, STORY_FILE_NAME)

    try:
        with open(original_story_path, "r", encoding='utf-8') as f: 
            original_narrative_text = f.read()
    except Exception as e:
        print(f" Original file read error {STORY_FILE_NAME}: {e}. Skip.")
        continue

    narrative_text = original_narrative_text 

    if REWRITE_MODE:
        print(f"\n--- Start of rewriting history {STORY_FILE_NAME} (LLM Stage 1) ---")
        try:
            rewritten_text = rewrite_story_for_image_creation(original_narrative_text)
            
            with open(rewritten_story_path, "w", encoding='utf-8') as f:
                f.write(rewritten_text)
            print(f"Story rewrited and saved in: {rewritten_story_path}")
            narrative_text = rewritten_text 
        except Exception as e:
            print(f" Error rewriting LLM: {e}. I continue with the original.")
            
    narrative_scenes = [s.strip() for s in narrative_text.split('\n\n') if s.strip()]
    if not narrative_scenes:
        print(f"WARNING: No valid scenes found. Skip.")
        continue

    story_output_folder = f"story_{story_number:02d}_images"
    os.makedirs(story_output_folder, exist_ok=True)

    generated_images = []
    previous_visual_prompts_summary = ""
    previous_image_path = "" 
    first_scene_prompt_for_story = "" 
    
    print(f"\n--- Start of VLM generation for {len(narrative_scenes)} scenes (LM Phase 2 + I2I) ---")

    
    for i, sentence in enumerate(narrative_scenes):
        scene_number = i + 1
        image_filename = os.path.join(story_output_folder, f"scene_{scene_number}.jpeg") 
        
        print(f"\n--- Scena {scene_number}/{len(narrative_scenes)}: \"{sentence[:50]}...\" ---")
        
        detailed_prompt = generate_visual_prompt(sentence, previous_scenes_context=previous_visual_prompts_summary)
        
        if scene_number == 1:
            first_scene_prompt_for_story = detailed_prompt 

        print(f"Final Detailed Prompt (used for VLM):\n{detailed_prompt[:150]}...")
            
        params = {
            "prompt": detailed_prompt,
            "output_format": "jpeg",
            "model": VLM_MODEL,
            "negative_prompt": NEGATIVE_PROMPT_GLOBAL,
            "seed": 0,
            "image": "",
            "mask": ""
        }
        
        if scene_number == 1:
            params["aspect_ratio"] = ASPECT_RATIO
            params["mode"] = "text-to-image"
        elif previous_image_path and os.path.exists(previous_image_path):
            params["image"] = previous_image_path
            params["mode"] = "image-to-image"
            params["strength"] = I2I_STRENGTH
            params.pop("aspect_ratio", None)
        else:
            params["aspect_ratio"] = ASPECT_RATIO
            params["mode"] = "text-to-image"
            
        try:
            response = send_generation_request(VLM_HOST, params)
            output_image = response.content
            
            with open(image_filename, "wb") as f:
                f.write(output_image)
                
            print(f" Image saved as {image_filename}")
            IPython.display.display(Image.open(BytesIO(output_image)))
            
            previous_image_path = image_filename
            generated_images.append(image_filename)
            previous_visual_prompts_summary += f"; Story {story_number:02d} Scene {scene_number} visual description: {detailed_prompt}" 
                
        except Exception as e:
            print(f" VLM generation error for the scene {scene_number}: {e}")
            break

    print(f"\n--- Pipeline completed for the story: {STORY_FILE_NAME} ---\n")

    all_stories_data.append({
        "story_number": story_number,
        "story_output_folder": story_output_folder,
        "generated_images": generated_images,
        "first_scene_prompt": first_scene_prompt_for_story
    })

print("\n--- All the pipeline completed ---\n")

# Metrics with Anomalies

In [ ]:
import torch
import lpips
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from torchmetrics.multimodal import CLIPImageQualityAssessment
from torchmetrics.multimodal.clip_score import CLIPScore
import os
import re
import glob
from transformers import AutoTokenizer 
import os


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_fn_alex = lpips.LPIPS(net='alex').to(device)

# CLIP-IQA Prompts we want to evaluate
IQA_PROMPTS = ('quality', 'sharpness') 

# Prompt to detect unwanted items in the scene
ANOMALY_PROMPTS = (
    'a cat', 
    'a tabby cat',  
    'an unexpected character',
    'a cat that should not be there',
    'text, watermark, low quality',
    'cat faces in the image',
    'characters with cat features',
    'characters that look like cats',
    'characters with feline traits',
    'characters with cat ears',
    'visible text',             
    'text on image',            
    'watermark',                
    'copyright notice',         
    'words on the screen',     
    'handwritten text',         
    'caption or label',         
    'letters or numbers'    
    
    
) 

clip_iqa_metric = CLIPImageQualityAssessment(
    model_name_or_path='openai/clip-vit-base-patch32', 
    data_range=255.0, 
    prompts=IQA_PROMPTS
).to(device)

clip_score_metric = CLIPScore(model_name_or_path="openai/clip-vit-base-patch32").to(device)
CLIP_TOKENIZER = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")

def load_and_preprocess_image(path: str, size: int = 224) -> torch.Tensor:
    """Load image, resize to (size,size) and return float32 tensor on device in 0..255 range."""
    img = Image.open(path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((size, size), interpolation=Image.BICUBIC),
        transforms.ToTensor(),  
    ])
    tensor = transform(img).mul(255.0).to(device, dtype=torch.float32)
    return tensor

def calculate_lpips_distance(image_tensor_1: torch.Tensor, image_tensor_2: torch.Tensor) -> float:
    """
    Compute LPIPS where inputs are expected in [-1,1].
    image_tensor_* are in 0..255 (C,H,W). Convert to [-1,1] then compute LPIPS.
    """
    # Convert to [-1,1]
    img1 = (image_tensor_1 / 127.5) - 1.0
    img2 = (image_tensor_2 / 127.5) - 1.0

    # Add batch dim and ensure device/dtype
    img1 = img1.unsqueeze(0).to(device, dtype=torch.float32)
    img2 = img2.unsqueeze(0).to(device, dtype=torch.float32)

    with torch.no_grad():
        lpips_dist = loss_fn_alex(img1, img2)
    return float(lpips_dist.squeeze().cpu().item())

#FIX TOKEN MAX OF 77
def calculate_clip_score(image_tensor: torch.Tensor, caption: str) -> float:
    tokenized_output = CLIP_TOKENIZER.encode_plus(
        caption,
        max_length=77,
        truncation=True,
        return_tensors='pt'
    )
    input_ids = tokenized_output['input_ids'][0]
    truncated_caption = CLIP_TOKENIZER.decode(input_ids, skip_special_tokens=True)
    
    image_input = image_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        score = clip_score_metric(image_input, [truncated_caption])
    return float(score.cpu().item())
    
def calculate_clip_iqa(images_tensor: torch.Tensor) -> dict:
    with torch.no_grad():
        scores = clip_iqa_metric(images_tensor)
    return {k: v.cpu().numpy() for k, v in scores.items()}

def calculate_anomaly_clip_score(image_tensor: torch.Tensor, anomaly_prompts: tuple) -> float:
   
    total_score = 0.0
    
    for prompt in anomaly_prompts:
        score = calculate_clip_score(image_tensor, prompt) 
        total_score += score
        
    avg_anomaly_score = total_score / len(anomaly_prompts)
    return avg_anomaly_score
    
def load_story_text(story_num: int):
    
    STORY_TEXT_PATH = # Directory path where original stories are stored example: "directory/of/original/stories"
    
    STORY_FILE_NAME = f"{story_num:02d}.txt" # Need to be called 01.txt, 02.txt, etc.
    narrative_text_path = os.path.join(STORY_TEXT_PATH, STORY_FILE_NAME)

    try:
        with open(narrative_text_path, "r", encoding='utf-8') as f:
            narrative_text = f.read()
            narrative_scenes = [s.strip() for s in narrative_text.split('\n\n') if s.strip()]
            return narrative_text, narrative_scenes
    except FileNotFoundError:
        return "", []
    except Exception as e:
        return "", []

cross_story_data = {
    'lpips': {},        
    'clip_score': {},   
    'anomaly_score': {},
    'clip_iqa': {p: {} for p in IQA_PROMPTS} 
}

In [ ]:
import os
import re
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt


required = [
    'load_and_preprocess_image',
    'calculate_lpips_distance',
    'calculate_clip_score',
    'calculate_clip_iqa',
    'calculate_anomaly_clip_score', 
    'device',
    'IQA_PROMPTS',
    'ANOMALY_PROMPTS', 
    'load_story_text' 
]
missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing required metric helpers or models: %s.\n"
        "Check the initialization of functions."
        % missing
    )

IMAGE_ROOT = os.getcwd() 
FOLDER_PATTERN = "story_*_images" 
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")

def _natural_sort_key(s):
    
    s = os.path.basename(s)
    parts = re.split(r'(\d+)', s)
    key = []
    for p in parts:
        if p.isdigit():
            key.append(int(p))
        else:
            key.append(p.lower())
    return key


def find_story_folders(root: str):
    candidates = glob.glob(os.path.join(root, FOLDER_PATTERN))
    dirs = [d for d in candidates if os.path.isdir(d)]
    dirs.sort(key=_natural_sort_key)
    return dirs


def find_images_in_folder(folder: str):
    files = []
    for fname in os.listdir(folder):
        ext = os.path.splitext(fname.lower())[1]
        if ext not in IMAGE_EXTS:
            continue
        name = os.path.splitext(fname)[0]
        if re.search(r'scene[_-]?\d+', name, flags=re.I):
            files.append(os.path.join(folder, fname))
        else:
            lower = name.lower()
            if any(token in lower for token in ('metric', 'metrics', 'plot')):
                continue
    files.sort(key=_natural_sort_key)
    return files


story_folders = find_story_folders(IMAGE_ROOT)
if not story_folders:
    print(f"No story folders found matching pattern '{FOLDER_PATTERN}' in {IMAGE_ROOT}")
else:
    print(f"Found {len(story_folders)} folders: {[os.path.basename(f) for f in story_folders]}")

for folder in story_folders:
    try:
        story_num_match = re.findall(r'\d+', os.path.basename(folder))
        story_num = int(story_num_match[0]) if story_num_match else None
        print(f"\n--- Processing story folder: {os.path.basename(folder)} ---")

        
        story_text, story_scenes = "", []
        if story_num is not None:
            try:
                story_text, story_scenes = load_story_text(story_num)
            except Exception:
                story_text, story_scenes = "", []
                
        if not story_text:
            PROMPT_REFERENCE = os.path.basename(folder)
            print(f"WARNING: Story text not found, use folder name as  {os.path.basename(folder)}.Use the folder name as prompt reference.")
        else:
            
            PROMPT_REFERENCE = story_scenes[0] if story_scenes else os.path.basename(folder)
            print(f"Story {story_num:02d} loaded: {len(story_scenes)} scenes")

        # Find images in the folder 
        images = find_images_in_folder(folder)
        if not images:
            print(f"No scene images found in {folder}. Skipping.")
            continue

        print(f"Found {len(images)} scene images -> computing metrics...")

        # Load and preprocess images
        all_tensors = []
        labels = []
        for idx, img_path in enumerate(images):
            try:
                t = load_and_preprocess_image(img_path)
                all_tensors.append(t)
                labels.append("T2I (Image0)" if idx == 0 else f"I2I (Image{idx})")
            except FileNotFoundError:
                print(f"File not found: {img_path}. Skipping image.")
                continue
            except Exception as e:
                print(f"Error loading image {img_path}: {e}. Skipping this image.")
                continue

        if not all_tensors:
            print("No valid images to process; skipping story.")
            continue

        
        
        ref_tensor = all_tensors[0]

        # LPIPS, CLIPScore, and ANOMALY_SCORE 
        lpips_results = {}
        clip_score_results = {}
        anomaly_score_results = {} 
        clip_iqa_score_results = {}
        
        for i, tensor in enumerate(all_tensors):
            try:
                # 1. LPIPS
                lpips_val = 0.0 if i == 0 else calculate_lpips_distance(tensor, ref_tensor)
                lpips_results[labels[i]] = lpips_val

                # 2. CLIP Score 
                scene_prompt = story_scenes[i] if i < len(story_scenes) else PROMPT_REFERENCE
                clip_val = calculate_clip_score(tensor, scene_prompt)
                clip_score_results[labels[i]] = clip_val
                
                # 3. Anomaly Score 
                anomaly_val = calculate_anomaly_clip_score(tensor, ANOMALY_PROMPTS)
                anomaly_score_results[labels[i]] = anomaly_val
                
                # 4. Average Score for all the images
                cross_story_data['lpips'].setdefault(i, []).append(lpips_val)
                cross_story_data['clip_score'].setdefault(i, []).append(clip_val)
                cross_story_data['anomaly_score'].setdefault(i, []).append(anomaly_val)
                
                
            except Exception as e:
                print(f"Error computing metrics for {labels[i]}: {e}")
                lpips_results[labels[i]] = 0.0
                clip_score_results[labels[i]] = 0.0
                anomaly_score_results[labels[i]] = 0.0 

        # CLIP-IQA 
        try:
            batch_images_tensor = torch.stack(all_tensors)
            clip_iqa_scores_dict = calculate_clip_iqa(batch_images_tensor)
            if clip_iqa_scores_dict is not None: 
                for i in range(len(labels)):
                    for prompt_key in IQA_PROMPTS:
                        score = clip_iqa_scores_dict[prompt_key][i]
                        cross_story_data['clip_iqa'][prompt_key].setdefault(i, []).append(score) 
        except Exception as e:
            print(f"Error computing CLIP-IQA: {e}")
            clip_iqa_scores_dict = None

        n_images = len(labels)
        n_prompts = len(IQA_PROMPTS)
        width = 0.8 / max(1, n_images)
        
        ordered_lpips = [lpips_results[name] for name in labels]
        i2i_lpips_values = ordered_lpips[1:] 
        
        # Calculate average LPIPS excluding T2I
        if i2i_lpips_values:
            avg_lpips = sum(i2i_lpips_values) / len(i2i_lpips_values)
            print(f"Average LPIPS for I2I images (excluding T2I): {avg_lpips:.4f}")
        else:
            avg_lpips = 0.0
            print("Only T2I image found or insufficient data. LPIPS average set to 0.0.")

        avg_clip = sum(clip_score_results.values()) / n_images
        avg_anomaly = sum(anomaly_score_results.values()) / n_images 

        
        final_names = labels + ["AVG RESULTS"]
        final_lpips_values = [lpips_results[name] for name in labels] + [avg_lpips]
        final_clip_values = [clip_score_results[name] for name in labels] + [avg_clip]
        final_anomaly_values = [anomaly_score_results[name] for name in labels] + [avg_anomaly]

        
        fig, axes = plt.subplots(ncols=4, figsize=(24, 6))
        plt.suptitle(f"Metrics - {os.path.basename(folder)} ({n_images} images)", fontsize=14, y=1.02)

        # LPIPS (Index 0)
        ax_lpips = axes[0]
        colors_lpips = ["#FF0000"] * n_images + ["#FFA500"]
        ax_lpips.bar(final_names, final_lpips_values, color=colors_lpips)
        ax_lpips.set_title("LPIPS (visual distance)")
        ax_lpips.set_ylabel("LPIPS (lower = better)")
        ax_lpips.axhline(y=0.1, color='purple', linestyle='--', linewidth=2)
        ax_lpips.text(0, 0.12, 'Identical threshold (0.10)', color='black', fontsize=10, fontweight='bold')
        ax_lpips.axhline(y=0.5, color='green', linestyle='--', linewidth=2)
        ax_lpips.text(0, 0.52, 'Similar threshold (0.50)', color='black', fontsize=10, fontweight='bold')
        ax_lpips.axhline(y=0.6, color='black', linestyle='--', linewidth=2)
        ax_lpips.text(0, 0.62, 'Different threshold (>0.60)', color='black', fontsize=10, fontweight='bold')
        ax_lpips.set_xticklabels(final_names, rotation=20, ha="right")
        ax_lpips.set_ylim(0, max(final_lpips_values) * 1.2 if max(final_lpips_values) > 0.65 else 0.65)
        for i, v in enumerate(final_lpips_values):
            ax_lpips.text(i, v + 0.01, f"{v:.4f}", ha="center")

        # CLIPScore (Index 1)
        ax_clip = axes[1]
        colors_clip = ["#0077B6"] * n_images + ["#00008B"]
        ax_clip.bar(final_names, final_clip_values, color=colors_clip)
        ax_clip.set_title("CLIPScore (text-image coherence)")
        ax_clip.set_ylabel("CLIPScore (higher = better)")
        ax_clip.axhline(y=25, color='red', linestyle='--', linewidth=2)
        ax_clip.text(0, 27, 'Accetable threshold (25)', color='black', fontsize=10, fontweight='bold')
        ax_clip.axhline(y=30, color='purple', linestyle='--', linewidth=2)
        ax_clip.text(0, 32, 'Optimal threshold (30)', color='black', fontsize=10, fontweight='bold')
        ax_clip.set_xticklabels(final_names, rotation=20, ha="right")
        ax_clip.set_ylim(0, max(final_clip_values) * 1.2 if max(final_clip_values) > 36 else 36)
        for i, v in enumerate(final_clip_values):
            ax_clip.text(i, v + 0.5, f"{v:.2f}", ha="center")
            
        # Anomaly Score (Index 2)
        ax_anomaly = axes[2]
        colors_anomaly = ["#9C3D54"] * n_images + ["#5B1D2F"]
        ax_anomaly.bar(final_names, final_anomaly_values, color=colors_anomaly)
        ax_anomaly.set_title("Anomaly Score")
        ax_anomaly.set_ylabel("Anomaly Score (higher = worse)")
        ax_anomaly.axhline(y=20, color='red', linestyle='--', linewidth=2)
        ax_anomaly.text(0, 22, 'Suspicion threshold (20)', color='black', fontsize=10, fontweight='bold')
        ax_anomaly.set_xticklabels(final_names, rotation=20, ha="right")
        ax_anomaly.set_ylim(0, max(final_anomaly_values) * 1.2 if max(final_anomaly_values) > 25 else 25)
        for i, v in enumerate(final_anomaly_values):
            ax_anomaly.text(i, v + 0.5, f"{v:.2f}", ha="center")


        # CLIP-IQA (Index 3)
        ax_iqa = axes[3]
        if clip_iqa_scores_dict is not None:
            x = np.arange(n_prompts)
            cmap = plt.cm.get_cmap("tab10", n_images)
            for i, lbl in enumerate(labels):
                vals = [clip_iqa_scores_dict[prompt][i] for prompt in IQA_PROMPTS]
                ax_iqa.bar(x + i * width, vals, width, label=lbl, color=cmap(i))
                for j, val in enumerate(vals):
                    ax_iqa.text(x[j] + i * width + width/2, val + 0.005, f"{val:.2f}", ha="center", va="bottom", fontsize=9)
            ax_iqa.set_xticks(x + width * (n_images - 1) / 2)
            ax_iqa.set_xticklabels([p.capitalize() for p in IQA_PROMPTS], rotation=20)
            ax_iqa.set_title("CLIP-IQA (perceptual quality)")
            ax_iqa.legend(loc="lower center", bbox_to_anchor=(0.5, -0.35), ncol=max(1, n_images))
            ax_iqa.set_ylim(0, min(1.0, max([max(clip_iqa_scores_dict[p]) for p in IQA_PROMPTS]) * 1.2))
        else:
            ax_iqa.text(0.5, 0.5, "CLIP-IQA not available", ha="center")
            ax_iqa.set_xticks([])
            ax_iqa.set_yticks([])

        
        plt.tight_layout(rect=[0, 0.1, 1, 0.95])

        #SAVE PLOT ON DISK
        out_plot = os.path.join(folder, f"story_{story_num:02d}_metrics_all_images_with_anomaly.png")
        plt.savefig(out_plot, dpi=150)
        plt.close(fig)
        print(f"Saved metrics plot: {out_plot}")

    except Exception as e:
        print(f"Error processing folder {folder}: {e}")
        continue

print("\n--- Analysis completed for all stories ---")

In [ ]:
# BLOCK FOR THE AVERAGE SCORE OF EACH IMAGE


import numpy as np
import matplotlib.pyplot as plt
import os
import re


def plot_cross_story_metrics(cross_story_data: dict, IQA_PROMPTS: tuple, IMAGE_ROOT: str):
    
    if not cross_story_data.get('lpips', {}):
        print("\nNOTICE: No data collected for cross-story analysis. Could not generate aggregate graph.")
        #WAR
        return

    print("\n--- Cross-Story Aggregate Chart Generation ---")

    
    max_scene_index = max(cross_story_data['lpips'].keys()) 
    max_scenes = max_scene_index + 1
    scene_labels = [f"Scene {i}" for i in range(max_scenes)]

    
    avg_cross_lpips = [np.mean(cross_story_data['lpips'].get(i, [0])) for i in range(max_scenes)]
    avg_cross_clip = [np.mean(cross_story_data['clip_score'].get(i, [0])) for i in range(max_scenes)]
    avg_cross_anomaly = [np.mean(cross_story_data['anomaly_score'].get(i, [0])) for i in range(max_scenes)]
    
    # CLIP-IQA
    avg_cross_iqa_data = {}
    for prompt in IQA_PROMPTS:
        avg_cross_iqa_data[prompt] = [
            np.mean(cross_story_data['clip_iqa'][prompt].get(i, [0])) 
            for i in range(max_scenes)
        ]

    
    fig, axes = plt.subplots(ncols=4, figsize=(24, 6))
    plt.suptitle("Overall Average Metrics (Cross-Story)", fontsize=16, y=1.02)
    
    
    # LPIPS Average (Axes 0)
    ax_lpips_avg = axes[0]
    ax_lpips_avg.bar(scene_labels, avg_cross_lpips, color="#FF0000")
    ax_lpips_avg.axhline(y=0.1, color='purple', linestyle='--', linewidth=2)
    ax_lpips_avg.text(0, 0.12, 'Identical threshold (0.10)', color='black', fontsize=10, fontweight='bold')
    ax_lpips_avg.axhline(y=0.5, color='green', linestyle='--', linewidth=2)        
    ax_lpips_avg.text(0, 0.52, 'Similar threshold (0.50)', color='black', fontsize=10, fontweight='bold')
    ax_lpips_avg.axhline(y=0.6, color='black', linestyle='--', linewidth=2)
    ax_lpips_avg.text(0, 0.62, 'Different threshold (>0.60)', color='black', fontsize=10, fontweight='bold')
    ax_lpips_avg.set_title("LPIPS Average (visual distance)")
    ax_lpips_avg.set_ylabel("LPIPS (lower = better)")
    ax_lpips_avg.set_xticklabels(scene_labels, rotation=20, ha="right")
    ax_lpips_avg.set_ylim(0, max(final_lpips_values) * 1.2 if max(final_lpips_values) > 0.65 else 0.65)

    for i, v in enumerate(avg_cross_lpips):
        ax_lpips_avg.text(i, v + 0.01, f"{v:.4f}", ha="center")
        
    # CLIPScore Average (Axes 1)
    ax_clip_avg = axes[1]
    ax_clip_avg.bar(scene_labels, avg_cross_clip, color="#0077B6")
    ax_clip_avg.set_title("CLIPScore Average (text-image coherence)")
    ax_clip_avg.set_ylabel("CLIPScore (higher = better)")
    ax_clip_avg.axhline(y=25, color='red', linestyle='--', linewidth=2)
    ax_clip_avg.text(0, 27, 'Accetable threshold (25)', color='black', fontsize=10, fontweight='bold')
    ax_clip_avg.axhline(y=30, color='purple', linestyle='--', linewidth=2)
    ax_clip_avg.text(0, 32, 'Optimal threshold (30)', color='black', fontsize=10, fontweight='bold')
    ax_clip_avg.set_xticklabels(scene_labels, rotation=20, ha="right")
    ax_clip_avg.set_ylim(0, max(final_clip_values) * 1.2 if max(final_clip_values) > 36 else 36)

    for i, v in enumerate(avg_cross_clip):
        ax_clip_avg.text(i, v + 0.5, f"{v:.2f}", ha="center")
        
    # Anomaly Score Average (Axes 2)
    ax_anomaly_avg = axes[2]
    ax_anomaly_avg.bar(scene_labels, avg_cross_anomaly, color="#9C3D54")
    ax_anomaly_avg.set_title("Anomaly Score Average")
    ax_anomaly_avg.set_ylabel("Anomaly Score (higher = worse)")
    ax_anomaly_avg.set_xticklabels(scene_labels, rotation=20, ha="right")
    ax_anomaly_avg.axhline(y=20, color='red', linestyle='--', linewidth=2)
    ax_anomaly_avg.text(0, 22, 'Suspicion threshold (20)', color='black', fontsize=10, fontweight='bold')
    ax_anomaly_avg.set_ylim(0, max(final_anomaly_values) * 1.2 if max(final_anomaly_values) > 25 else 25)

    for i, v in enumerate(avg_cross_anomaly):
        ax_anomaly_avg.text(i, v + 0.5, f"{v:.2f}", ha="center")
        
    # CLIP-IQA Average (Axes 3)
    ax_iqa_avg = axes[3]
    
    n_prompts_iqa = len(IQA_PROMPTS)
    width_iqa = 0.8 / max(1, n_prompts_iqa) 
    x_positions = np.arange(len(scene_labels)) 

    colors_iqa = plt.cm.get_cmap("viridis", n_prompts_iqa) 

    max_overall_iqa_val = 0.0

    for p_idx, prompt_key in enumerate(IQA_PROMPTS):
        avg_iqa_scores = avg_cross_iqa_data[prompt_key]
        
        ax_iqa_avg.bar(x_positions + p_idx * width_iqa - (n_prompts_iqa - 1) * width_iqa / 2, 
                       avg_iqa_scores, 
                       width_iqa, 
                       label=prompt_key.capitalize(), 
                       color=colors_iqa(p_idx))
        
        for i, v in enumerate(avg_iqa_scores):
            ax_iqa_avg.text(x_positions[i] + p_idx * width_iqa - (n_prompts_iqa - 1) * width_iqa / 2, 
                           v + 0.01, 
                           f"{v:.2f}", 
                           ha="center", 
                           va="bottom", 
                           fontsize=8,
                           color=colors_iqa(p_idx)) 
        
        if avg_iqa_scores:
            max_overall_iqa_val = max(max_overall_iqa_val, max(avg_iqa_scores))

    ax_iqa_avg.set_title("CLIP-IQA Average (perceptual quality)")
    ax_iqa_avg.set_ylabel("CLIP-IQA Score (higher = better)")
    ax_iqa_avg.set_xticks(x_positions) 
    ax_iqa_avg.set_xticklabels(scene_labels, rotation=20, ha="right")
    ax_iqa_avg.legend(loc="lower center", bbox_to_anchor=(0.5, -0.35), ncol=n_prompts_iqa) 
    
    ax_iqa_avg.set_ylim(0, max(1.0, max_overall_iqa_val * 1.2)) 

    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    
    # Saving the PLOT WITH ANOMALIES
    out_plot_avg = os.path.join(IMAGE_ROOT, "all_stories_aggregate_metrics_with_anomalies.png")
    plt.savefig(out_plot_avg, dpi=150)
    plt.close(fig)
    print(f"\nSaved aggregate cross-story plot: {out_plot_avg}")



if 'cross_story_data' in globals() and cross_story_data:
    plot_cross_story_metrics(cross_story_data, IQA_PROMPTS, IMAGE_ROOT)
else:
    print("\nERROR: Cross_story_data variable was not found or is empty. Make sure you ran blocks 1 and 2 first.")